#### 09. Final models and the test year

The settings chosen on 2023 in notebook 07 are fixed here. Both models are refitted on 2017–2023, the full training table plus the validation year with every row at weight 1, and so are the threshold mapper and the baselines. Everything is then scored once on 2024, the test year, which no earlier notebook has used.

**Setup**

In [1]:
%load_ext autoreload
%autoreload 2
import time
import sys
sys.path.insert(0, "../code")
import pandas as pd
from pyspark.sql import functions as F

import config as C
import evaluate as E
import mapper as M
import models as MD
import baselines as B

spark = C.get_spark("task2-final")

**Final settings**

Taken from notebook 07 (LR tuning and random forest screening on 2023) and from notebook 04 for the climatology shrinkage.

In [2]:
TIERS = ("t0", "t1", "t2", "t3")
LR_SPEC = {"reg_param": 1e-6, "nonlinear": True}
RF_SPEC = {"max_depth": 24, "min_instances": 50, "feature_subset": "onethird", "max_bins": 256, "impurity": "entropy"}
NUM_TREES = C.RF_TREES
CLIM_M = 3000
REUSE = ('LR',)  # e.g. ('LR',): load that model from models/ instead of refitting it

**Data**

In [3]:
train_full = spark.read.parquet(str(C.ASSEMBLED / "train_full"))
val = spark.read.parquet(str(C.ASSEMBLED / "val"))
fit = train_full.unionByName(val).cache()
test = spark.read.parquet(str(C.ASSEMBLED / "test")).cache()
FEATURES = MD.feature_columns(fit, TIERS)
print(f"fit 2017-{C.VAL_YEAR}: {fit.count():,} rows   test {C.TEST_YEAR}: {test.count():,} rows   {len(FEATURES)} features")

fit 2017-2023: 41,137,290 rows   test 2024: 6,374,476 rows   58 features


**Threshold mapper**

In [4]:
mp = M.fit(fit)
M.save(mp, C.MODELS / "mapper.json")
{k: (len(v["y"]), max(v["y"])) for k, v in mp["classes"].items()}

{1: (1, 0.2),
 2: (1, 0.4),
 3: (3, 1.0),
 4: (5, 2.0),
 5: (27, 5.0),
 6: (50, 10.0),
 7: (100, 20.0),
 8: (416, 71.8)}

**Final models**

In [5]:
from pyspark.ml import PipelineModel

final = {}
for kind, build in [("LR", lambda: MD.build_lr(FEATURES, **LR_SPEC)),
                    ("RF", lambda: MD.build_rf(FEATURES, num_trees=NUM_TREES, **RF_SPEC))]:
    path, start = str(C.MODELS / kind.lower()), time.time()
    if kind in REUSE:
        final[kind] = PipelineModel.load(path)
    else:
        final[kind] = build().fit(fit)
        final[kind].write().overwrite().save(path)
    print(kind, "loaded" if kind in REUSE else "fitted and saved", f"in {(time.time() - start) / 60:.0f} min")
print("RF nodes:", final["RF"].stages[-1].totalNumNodes)

LR loaded in 0 min


26/10/14 11:10:43 ERROR Utils: Uncaught exception in thread executor-heartbeater
java.lang.NullPointerException: Cannot invoke "java.util.concurrent.ConcurrentHashMap.isEmpty()" because the return value of "org.apache.spark.sql.util.PartitionKeyedAccumulator.byPartition()" is null
	at org.apache.spark.sql.util.PartitionKeyedAccumulator.isZero(PartitionKeyedAccumulator.scala:51)
	at org.apache.spark.executor.Executor.$anonfun$reportHeartBeat$2(Executor.scala:1536)
	at org.apache.spark.executor.Executor.$anonfun$reportHeartBeat$2$adapted(Executor.scala:1536)
	at scala.collection.immutable.Vector.filterImpl(Vector.scala:166)
	at scala.collection.immutable.Vector.filterImpl(Vector.scala:116)
	at scala.collection.StrictOptimizedIterableOps.filterNot(StrictOptimizedIterableOps.scala:220)
	at scala.collection.StrictOptimizedIterableOps.filterNot$(StrictOptimizedIterableOps.scala:220)
	at scala.collection.immutable.Vector.filterNot(Vector.scala:116)
	at org.apache.spark.executor.Executor.$anon

RF fitted and saved in 589 min
RF nodes: 1425442


**Baselines on the same years**

The constant forecast takes the class shares of 2017–2023, and climatology and persistence are refitted on the same table, which holds every labelled row of those years at weight 1.

In [6]:
shares = fit.groupBy("label").agg(F.sum("w").alias("sw")).toPandas().set_index("label").sw.sort_index()
shares = shares / shares.sum()
from pyspark.ml.functions import array_to_vector
forecasts = {
    "constant": test.withColumn("probability", array_to_vector(F.array(*[F.lit(float(p)) for p in shares]))),
    "climatology": B.predict_climatology(B.fit_climatology(fit, CLIM_M), test),
    "persistence": B.predict_persistence(B.fit_persistence(fit), test),
    **{f"{kind} {'+'.join(t.upper() for t in TIERS)}": model.transform(test) for kind, model in final.items()},
}

**Results on 2024**

In [7]:
rows = []
for name, fc in forecasts.items():
    fc = fc.select("label", "y_mm", "w", "regime", "probability").cache()
    rows.append({"model": name, "subset": "all", **E.score(fc)})
    for regime in ["dry_now", "wet_now"]:
        rows.append({"model": name, "subset": regime, **E.score(fc.filter(F.col("regime") == regime))})
    fc.unpersist()
results = pd.DataFrame(rows).set_index(["subset", "model"]).sort_index()
results.to_csv(C.RESULTS / "models_test.csv")
results.loc["all"].sort_values("RPS").round(4)

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


,RPS,log loss,Brier 0,Brier 1,Brier 5,Brier 20,AUC-PR 0,AUC-PR 1,AUC-PR 5,AUC-PR 20
model,,,,,,,,,,
RF T0+T1+T2+T3,0.0887,0.1863,0.0248,0.0121,0.0054,0.0009,0.8015,0.7117,0.5246,0.2285
LR T0+T1+T2+T3,0.0922,0.1933,0.0259,0.0126,0.0055,0.0009,0.7850,0.6888,0.5066,0.1956
persistence,0.1246,0.2577,0.0355,0.0170,0.0070,0.0010,0.6406,0.4731,0.2500,0.0495
climatology,0.1864,0.3437,0.0578,0.0240,0.0082,0.0010,0.1254,0.0540,0.0234,0.0043
constant,0.1903,0.3545,0.0596,0.0244,0.0082,0.0010,0.0636,0.0250,0.0083,0.0010


In [8]:
cols = ["RPS", "log loss", "Brier 0", "AUC-PR 0", "AUC-PR 5"]
pd.concat({"dry now": results.loc["dry_now"][cols], "wet now": results.loc["wet_now"][cols]}, axis=1).sort_values(("dry now", "RPS")).round(4)

dry now                                    wet now           \
                   RPS log loss Brier 0 AUC-PR 0 AUC-PR 5     RPS log loss   
model                                                                        
RF T0+T1+T2+T3  0.0530   0.1111  0.0182   0.4298   0.1903  0.6140   1.2924   
LR T0+T1+T2+T3  0.0544   0.1142  0.0187   0.3962   0.1616  0.6487   1.3581   
persistence     0.0679   0.1611  0.0247   0.0253   0.0027  0.9580   1.6771   
constant        0.0708   0.1760  0.0260   0.0253   0.0027  1.9499   2.9833   
climatology     0.0710   0.1705  0.0259   0.0522   0.0084  1.8859   2.8948   

                                          
               Brier 0 AUC-PR 0 AUC-PR 5  
model                                     
RF T0+T1+T2+T3  0.1219   0.9427   0.6466  
LR T0+T1+T2+T3  0.1324   0.9324   0.6352  
persistence     0.1935   0.8230   0.3401  
constant        0.5544   0.6275   0.0907  
climatology     0.5278   0.6403   0.1204

**Thresholds between the class edges**

The threshold mapper turns the class probabilities into P(Y > θ) for any θ. The Brier score at 3, 15 and 30 mm, which are not class edges, checks it on the test year.

In [9]:
brier = {}
for name, fc in forecasts.items():
    df = fc.select("y_mm", "w", "probability")
    for theta in C.EXTRA_THRESHOLDS:
        df = M.add_threshold(df, mp, theta)
    df = df.cache()
    brier[name] = {f"Brier {t:g}": E.brier_at(df, t) for t in C.EXTRA_THRESHOLDS}
    df.unpersist()
pd.DataFrame(brier).T.round(5)

,Brier 3,Brier 15,Brier 30
constant,0.01230,0.00194,0.00029
climatology,0.01220,0.00194,0.00029
persistence,0.00991,0.00184,0.00029
LR T0+T1+T2+T3,0.00757,0.00155,0.00027
RF T0+T1+T2+T3,0.00744,0.00156,0.00028


**Calibration on 2024**

In [10]:
tables = {}
for kind, model in final.items():
    ex = E.add_exceedance(model.transform(test))
    for th in [0.0, 5.0]:
        tables[f"{kind}, θ = {th:g} mm"] = E.reliability(ex, th).set_index("bin")[["n", "mean_forecast", "observed"]]
pd.concat(tables, axis=1).round(3)

LR, θ = 0 mm                        LR, θ = 5 mm                         \
               n mean_forecast observed            n mean_forecast observed   
bin                                                                           
0      5673626.0         0.010    0.010    6260022.0         0.002    0.003   
1       186433.0         0.143    0.152      47129.0         0.142    0.145   
2        96012.0         0.246    0.259      21338.0         0.245    0.245   
3        65661.0         0.348    0.364      12914.0         0.347    0.342   
4        50559.0         0.448    0.456       9045.0         0.447    0.436   
5        42322.0         0.549    0.546       6705.0         0.548    0.519   
6        39358.0         0.650    0.641       5649.0         0.649    0.632   
7        38872.0         0.750    0.729       4392.0         0.748    0.724   
8        42144.0         0.851    0.826       3992.0         0.849    0.831   
9       139489.0         0.975    0.962       3290.0         0.944    0.932   

    RF, θ = 0 mm                        RF, θ = 5 mm                         
               n mean_forecast observed            n mean_forecast observed  
bin                                                                          
0      5635189.0         0.010    0.009    6252948.0         0.003    0.002  
1       201344.0         0.144    0.134      53874.0         0.142    0.141  
2       110210.0         0.246    0.239      23620.0         0.245    0.257  
3        72795.0         0.347    0.351      13319.0         0.346    0.364  
4        55038.0         0.448    0.461       9082.0         0.447    0.469  
5        45446.0         0.549    0.571       6548.0         0.548    0.565  
6        40119.0         0.649    0.672       5186.0         0.648    0.676  
7        39569.0         0.750    0.775       4331.0         0.748    0.773  
8        42630.0         0.852    0.867       3669.0         0.849    0.880  
9       132136.0         0.971    0.976       1899.0         0.933    0.966

**Feature importance of the final forest**

In [11]:
importance = pd.Series(final["RF"].stages[-1].featureImportances.toArray(), index=FEATURES, name="importance")
importance.sort_values(ascending=False).head(15).round(4)

t2_idw_rain_15           0.2610
t2_idw_rain_30           0.1474
t2_wet_frac_10km         0.1221
t2_dist_rain_km          0.0775
t1_rain_5                0.0557
t2_island_rain_30        0.0505
t2_island_wet_frac       0.0488
t1_rain_15               0.0289
t2_far_rain_30           0.0264
t1_min_since_rain        0.0236
t2_idw_trend             0.0162
t2_wet_frac_5km          0.0137
t3_d15_idw_rain_30       0.0108
t3_d15_island_rain_30    0.0099
t1_rain_30               0.0094
Name: importance, dtype: float64